In [23]:
import pandas as pd
import numpy as np
from scipy.special import comb
from scipy.stats import binom

In [24]:
posses = pd.read_csv("possessive.tsv", sep="\t")
# фильтруем только семитские языки
semitic_p = posses[posses["family"] == "Afro-Asiatic"]
nilotic_p = posses[posses["family"] == "Eastern Sudanic"]

In [25]:
semitic_p

,wals code,name,value,description,latitude,longitude,genus,family,area
9,aeg,Arabic (Egyptian),4,No marking,30.000000,31.000000,Semitic,Afro-Asiatic,Morphology
21,bej,Beja,2,Dependent marking,18.000000,36.000000,Beja,Afro-Asiatic,Morphology
23,bma,Berber (Middle Atlas),2,Dependent marking,33.000000,-5.000000,Berber,Afro-Asiatic,Morphology
47,diz,Dizi,3,Double marking,6.166667,36.500000,Dizoid,Afro-Asiatic,Morphology
73,hau,Hausa,2,Dependent marking,12.000000,7.000000,West Chadic,Afro-Asiatic,Morphology
74,heb,Hebrew (Modern),2,Dependent marking,31.500000,34.833333,Semitic,Afro-Asiatic,Morphology
162,orh,Oromo (Harar),5,Other,9.000000,42.000000,Lowland East Cushitic,Afro-Asiatic,Morphology


In [26]:
nilotic_p

,wals code,name,value,description,latitude,longitude,genus,family,area
84,ik,Ik,2,Dependent marking,3.750000,34.166667,Kuliak,Eastern Sudanic,Morphology
119,lan,Lango,1,Head marking,2.166667,33.000000,Western Nilotic,Eastern Sudanic,Morphology
147,nar,Nara (in Ethiopia),4,No marking,15.083333,37.583333,Nara,Eastern Sudanic,Morphology
156,nbd,Nubian (Dongolese),2,Dependent marking,18.250000,30.750000,Nubian,Eastern Sudanic,Morphology


In [27]:
def haversine(lat1, lon1, lat2, lon2):
    """Функция для рассчёта расстояния на шаре"""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371  # Радиус Земли в километрах
    return c * r

In [28]:
df = pd.concat([semitic_p, nilotic_p], ignore_index=True)

k = df['value'].nunique()
print(f"Число значений признака (k): {k}\n")

family_stats = {}

for family_name, group in df.groupby('family'):
    n = len(group)
    if n < 2:
        print(f"Семья {family_name} содержит слишком мало языков ({n}) для расчета.")
        continue
    
    val_counts = group['value'].value_counts()
    r = val_counts.max()
    
    # МЕТРИКА B
    metric_B_family = r / n
    
    # МЕТРИКА A
    p_g = binom.sf(r - 1, n, 1 / k)
    metric_A_family = 1 - p_g
    
    P_g = comb(n, 2)
    C_g = sum(comb(v, 2) for v in val_counts if v > 1)
    
    family_stats[family_name] = {
        'n': n, 'r': r, 
        'metric_A': metric_A_family, 'metric_B': metric_B_family,
        'P_g': P_g, 'C_g': C_g
    }
    
    print(f"Семья: {family_name}")
    print(f"Количество языков (n): {n}")
    print(f"Частота мажоритарного значения (r): {r}")
    print(f"Метрика A (индивидуальная): {metric_A_family:.4f}")
    print(f"Метрика B (доля стабильности): {metric_B_family:.4f}\n")


sum_weighted_similarity = 0
sum_weights = 0

for f_name, stats in family_stats.items():
    weight = np.sqrt(stats['P_g'])
    similarity = stats['C_g'] / stats['P_g']
    
    sum_weighted_similarity += weight * similarity
    sum_weights += weight

R = sum_weighted_similarity / sum_weights
print(f"Общее внутриродственное сходство семей (R): {R:.4f}")

semitic_p['_key'] = 1
nilotic_p['_key'] = 1
unrelated_pairs = pd.merge(semitic_p, nilotic_p, on='_key', suffixes=('_sem', '_nil'))

unrelated_pairs['distance'] = haversine(
    unrelated_pairs['latitude_sem'], unrelated_pairs['longitude_sem'],
    unrelated_pairs['latitude_nil'], unrelated_pairs['longitude_nil']
)

pairs_within_5k = unrelated_pairs[unrelated_pairs['distance'] <= 5000]
P_unrelated = len(pairs_within_5k)

if P_unrelated > 0:
    C_unrelated = (pairs_within_5k['value_sem'] == pairs_within_5k['value_nil']).sum()
    U = C_unrelated / P_unrelated
    print(f"Всего неродственных пар в радиусе 5000 км (P_unrelated): {P_unrelated}")
    print(f"Из них совпали по значению (C_unrelated): {C_unrelated}")
    print(f"Базовое сходство неродственных языков (U): {U:.4f}")
else:
    U = 0
    print("Внимание: Не найдено пар неродственных языков в радиусе 5000 км.")

if U < 1:
    S_C = (R - U) / (1 - U)
    print(f"\nИТОГОВАЯ СТАБИЛЬНОСТЬ ПО МЕТРИКЕ C (S_C): {S_C:.4f}")
else:
    print("\nМетрику C рассчитать невозможно (U >= 1).")

semitic_p.drop(columns=['_key'], inplace=True, errors='ignore')
nilotic_p.drop(columns=['_key'], inplace=True, errors='ignore')

Число значений признака (k): 5

Семья: Afro-Asiatic
Количество языков (n): 7
Частота мажоритарного значения (r): 4
Метрика A (индивидуальная): 0.9667
Метрика B (доля стабильности): 0.5714

Семья: Eastern Sudanic
Количество языков (n): 4
Частота мажоритарного значения (r): 2
Метрика A (индивидуальная): 0.8192
Метрика B (доля стабильности): 0.5000

Общее внутриродственное сходство семей (R): 0.2442
Всего неродственных пар в радиусе 5000 км (P_unrelated): 26
Из них совпали по значению (C_unrelated): 8
Базовое сходство неродственных языков (U): 0.3077

ИТОГОВАЯ СТАБИЛЬНОСТЬ ПО МЕТРИКЕ C (S_C): -0.0916
